# 10 — Evidence-Grounded Prompting and RAG Interfaces

## Scenario
Northstar must answer questions about refund policies. We want to avoid hallucination, so we require the model to ground its answers in factual evidence.

**The Problem:** LLMs are eager to please and will often invent plausible-sounding policies if they don't know the answer.

In [ ]:
import os
from google import genai
from google.genai import types

# Initialize the client (requires GEMINI_API_KEY environment variable)
client = genai.Client()
MODEL_ID = 'gemini-2.5-flash'

USER_QUESTION = "What is the return policy for a custom-engraved Northstar mug?"


## Step 1: Ungrounded Generation (Baseline)

Watch what happens when we ask a niche question without any grounding.

In [ ]:
prompt = f"""You are a helpful customer support bot for Northstar.
Answer the following customer question:
{USER_QUESTION}
"""

response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt
)
print("--- Ungrounded Output ---")
print(response.text)

# Notice: The model likely hallucinates a plausible, but entirely invented, return policy 
# (e.g., "Custom items cannot be returned unless defective").

## Step 2: Manual Grounding (Classic RAG)

We "retrieve" a document (mocked here) and strictly instruct the model to use it.

In [ ]:
# Mock retrieval from a vector database
RETRIEVED_DOCUMENT = """
[Document ID: POL-992]
Title: Return Policy for Custom Items
Content: All custom-engraved Northstar items are final sale. We do not accept returns or exchanges on these items under any circumstances, even if defective, due to the bespoke nature of the engraving process.
"""

grounded_prompt = f"""You are a strict customer support bot for Northstar.
You must answer the user's question using ONLY the provided Reference Document.
If the Reference Document does not contain the answer, you must output "I don't know."

<reference_document>
{RETRIEVED_DOCUMENT}
</reference_document>

User Question: {USER_QUESTION}
"""

response_grounded = client.models.generate_content(
    model=MODEL_ID,
    contents=grounded_prompt
)
print("--- Manual Grounding Output ---")
print(response_grounded.text)

# Notice: The model successfully restricts its answer to the provided text.

## Step 3: Managed Grounding with Google Search (State of the Art)

Instead of manually pasting text into prompts, modern APIs natively support grounding endpoints. Here we demonstrate using Google Search natively via the Gemini API to ground an answer and receive structured citations.

In [ ]:
# We ask a real-world question to demonstrate the native Google Search grounding
real_world_question = "What is the current stock price of Alphabet (GOOG)?"

# We configure the model to use the Google Search tool
search_tool = types.Tool(
    google_search=types.GoogleSearch()
)

response_search = client.models.generate_content(
    model=MODEL_ID,
    contents=real_world_question,
    config=types.GenerateContentConfig(
        tools=[search_tool],
        temperature=0.0
    )
)

print("--- Managed Grounding (Google Search) Output ---")
print(response_search.text)

# The model now automatically retrieves data, reads it, and formats the answer with verifiable citations.